### PHASE 1: DATA ARCHITECTURE (The 5-Day Lifecycle)


In [1]:
import pandas as pd
import numpy as np

def generate_lifecycle_data(samples=1000000):
    np.random.seed(42)
    print(f"Generating {samples} player lifecycles...")
    
    data = {
        'player_id': range(samples),
        # Day 1 - Initial Exploration
        'd1_session_duration': np.random.uniform(2, 30, samples),
        'd1_shop_opens': np.random.randint(0, 5, samples),
        'd1_ads_watched': np.random.randint(0, 10, samples),
        
        # Day 3 - Behavioral Evolution
        'd3_cumulative_ads': np.random.randint(5, 30, samples),
        'd3_iap_intent_clicks': np.random.randint(0, 3, samples),
        'd3_retention_score': np.random.uniform(0, 1, samples),
        
        # Day 5 - The Ground Truth
        'd5_actual_purchase_made': np.random.choice([0, 1], samples, p=[0.96, 0.04]),
        'd5_engagement_velocity': np.random.uniform(0.5, 2.0, samples)
    }
    
    df = pd.DataFrame(data)
    
    # INITIAL SCORING LOGIC (For Flowchart 1)
    # High shop opens = Payer Trait | High ads = Watcher Trait
    d1_score = (df['d1_shop_opens'] * 3) - (df['d1_ads_watched'] * 1.5)
    
    df['d1_class'] = 1 # Default Neutral
    df.loc[d1_score > 4, 'd1_class'] = 2  # Payer
    df.loc[d1_score < -5, 'd1_class'] = 0 # Watcher
    
    return df

df_p2 = generate_lifecycle_data()
print("\nPHASE 1: Day 1 Baseline Established.")
print(df_p2['d1_class'].value_counts().rename({0:'Watcher', 1:'Neutral', 2:'Payer'}))

Generating 1000000 player lifecycles...

PHASE 1: Day 1 Baseline Established.
d1_class
Neutral    520456
Payer      239790
Watcher    239754
Name: count, dtype: int64


### PHASE 2: THE DAY 3 SENTINEL (XGBOOST RE-EVALUATION)


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb

# 1. UPDATING THE DATASET WITH YOUR SPECIFIC DAY 3 TRAITS
# We simulate the traits you described: IAP attempts, Avatar clicks, and Rage Quits.
np.random.seed(42)
samples = len(df_p2)

# New Behavioral Traits for Day 3
df_p2['d3_iap_attempts'] = np.random.choice([0, 1, 2], samples, p=[0.90, 0.08, 0.02])
df_p2['d3_rv_watched'] = np.random.randint(0, 15, samples)
df_p2['d3_avatar_clicks'] = np.random.randint(0, 10, samples)
df_p2['d3_ad_rage_quits'] = np.random.choice([0, 1], samples, p=[0.95, 0.05])

# 2. DEFINE THE TRANSITION LOGIC (As per your description)
# We use these to create 'Labels' for the XGBoost to learn from.

# Neutral -> Payer Logic
payer_idx = (df_p2['d1_class'] == 1) & ((df_p2['d3_iap_attempts'] > 0) | (df_p2['d1_shop_opens'] > 3))
df_p2.loc[payer_idx, 'target_d3'] = 2

# Neutral -> Watcher Logic
watcher_idx = (df_p2['d1_class'] == 1) & (df_p2['d3_rv_watched'] > 8)
df_p2.loc[watcher_idx, 'target_d3'] = 0

# Watcher -> Neutral (The Rage Quit Logic)
rage_quit_idx = (df_p2['d1_class'] == 0) & (df_p2['d3_ad_rage_quits'] > 0)
df_p2.loc[rage_quit_idx, 'target_d3'] = 1

# If no change, keep original class
df_p2['target_d3'] = df_p2['target_d3'].fillna(df_p2['d1_class'])

# 3. THE XGBOOST RE-CLASSIFIER
features_d3 = ['d1_shop_opens', 'd1_ads_watched', 'd3_iap_attempts', 'd3_rv_watched', 'd3_avatar_clicks', 'd3_ad_rage_quits']
X = df_p2[features_d3]
y = df_p2['target_d3']

# Training the model to recognize these transitions
model_d3 = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5)
model_d3.fit(X, y)

# 4. EXECUTE RE-CLASSIFICATION
df_p2['d3_final_class'] = model_d3.predict(X)

print("PHASE 2: Day 3 Re-evaluation Complete.")
print("\nNew Classification Counts (Day 3):")
print(df_p2['d3_final_class'].value_counts().rename({0:'Watcher', 1:'Neutral', 2:'Payer'}))

# Check how many 'Watchers' were saved back to 'Neutral'
saved_watchers = len(df_p2[(df_p2['d1_class'] == 0) & (df_p2['d3_final_class'] == 1)])
print(f"\n[ALERT]: {saved_watchers} Watchers moved back to Neutral due to Ad-Rage Quits.")

PHASE 2: Day 3 Re-evaluation Complete.

New Classification Counts (Day 3):
d3_final_class
Watcher    435785
Payer      314444
Neutral    249771
Name: count, dtype: int64

[ALERT]: 11939 Watchers moved back to Neutral due to Ad-Rage Quits.


### PHASE 3: THE DAY 5 "FINAL GATE" & PERMANENT LOCK


In [3]:
# 1. THE FINAL GATE LOGIC
df_p2['final_locked_class'] = df_p2['d3_final_class']

# CASE A: THE PAYER AUDIT
# If we called them a Payer (2) but they haven't spent by Day 5, we RECALIBRATE.
recalibrate_idx = (df_p2['d3_final_class'] == 2) & (df_p2['d5_actual_purchase_made'] == 0)

# For those being recalibrated, we check their 'Engagement Velocity'
# High velocity (Still playing) -> Neutral | Low velocity -> Watcher
df_p2.loc[recalibrate_idx & (df_p2['d5_engagement_velocity'] > 1.2), 'final_locked_class'] = 1 # Back to Neutral
df_p2.loc[recalibrate_idx & (df_p2['d5_engagement_velocity'] <= 1.2), 'final_locked_class'] = 0 # Back to Watcher

# CASE B: THE PERMANENT PAYER
# If they actually bought, they are locked as Payers (2) forever.
df_p2.loc[df_p2['d5_actual_purchase_made'] == 1, 'final_locked_class'] = 2

# CASE C: THE NEUTRAL DEFAULT
# If still Neutral by Day 5 and no purchase, we default them to Watcher to start recouping costs.
df_p2.loc[(df_p2['final_locked_class'] == 1) & (df_p2['d5_actual_purchase_made'] == 0), 'final_locked_class'] = 0

print("--- PROJECT 2: THE FINAL REVENUE AUDIT ---")
print(df_p2['final_locked_class'].value_counts().rename({0:'Watcher (Perm)', 1:'Neutral (Rare/Active)', 2:'Payer (Confirmed)'}))

# CALCULATING THE SUCCESS OF THE FUNNEL
confirmed_payers = len(df_p2[df_p2['final_locked_class'] == 2])
print(f"\n[ECONOMY REPORT]: We have {confirmed_payers} Confirmed High-Value Payers.")
print("All other users are now optimized for Ad-Revenue to maximize LTV.")

--- PROJECT 2: THE FINAL REVENUE AUDIT ---
final_locked_class
Watcher (Perm)       959779
Payer (Confirmed)     40221
Name: count, dtype: int64

[ECONOMY REPORT]: We have 40221 Confirmed High-Value Payers.
All other users are now optimized for Ad-Revenue to maximize LTV.
